# 04 - Experimental Prioritization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/cryptic-ip-pipeline/blob/main/notebooks/04_experimental_prioritization_colab.ipynb)

Builds the experimental follow-up artefacts (mutagenesis plan, DSF
plan, ranked summary) for the top N candidates from a completed
screen. As with notebook 03, you can either point this at an existing
screen on Drive or run a tiny on-the-fly screen so the rest of the
notebook has data.


## Run this first — fresh-Colab setup

Configure the variables below, then run every cell in this section in
order. Re-running is safe: the clone is updated in place and pip
installs are idempotent.

- `REPO_URL` / `BRANCH` — where to fetch the code from.
- `PROJECT_DIR` — where to clone into (`/content/...` lives on the
  Colab VM and is wiped between sessions).
- `MOUNT_DRIVE` — set `True` to mount Google Drive so outputs persist.
- `DRIVE_RESULTS` — if mounting, where on Drive to put `results/`.


In [ ]:
REPO_URL    = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
BRANCH      = 'main'
PROJECT_DIR = '/content/cryptic-ip-pipeline'
MOUNT_DRIVE = False                  # True to mount Google Drive
DRIVE_ROOT  = '/content/drive'
DRIVE_RESULTS = '/content/drive/MyDrive/crypticip/results'  # used only if MOUNT_DRIVE


In [ ]:
import os, subprocess, sys, pathlib

project = pathlib.Path(PROJECT_DIR)
project.parent.mkdir(parents=True, exist_ok=True)

def sh(cmd, check=True):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd, check=check)

if (project / '.git').exists():
    sh(['git', '-C', str(project), 'fetch', '--quiet', 'origin', BRANCH])
    sh(['git', '-C', str(project), 'checkout', BRANCH])
    sh(['git', '-C', str(project), 'reset', '--hard', f'origin/{BRANCH}'])
else:
    sh(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, str(project)])

os.chdir(project)
print('cwd:', os.getcwd())
sh([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'nbformat'])


In [ ]:
# Optional: mount Drive and redirect results/ onto it so outputs survive
# a runtime swap. Safe to re-run; pre-existing local results/ is backed
# up to a timestamped sibling rather than deleted.
import datetime, pathlib, shutil

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount(DRIVE_ROOT)
    drive_path = pathlib.Path(DRIVE_RESULTS)
    drive_path.mkdir(parents=True, exist_ok=True)
    target = pathlib.Path(PROJECT_DIR) / 'results'
    if target.is_symlink():
        target.unlink()
    elif target.exists():
        backup = target.with_name(f'results.local_backup_{datetime.datetime.now():%Y%m%d_%H%M%S}')
        shutil.move(str(target), str(backup))
        print('existing results/ backed up to:', backup)
    target.symlink_to(drive_path, target_is_directory=True)
    print('results/ ->', drive_path)
else:
    print('MOUNT_DRIVE=False; outputs will live on the Colab VM only.')


In [ ]:
# Verify the CLI is wired up correctly.
!crypticip --version
!crypticip check-env
!crypticip list-configs


## Locate (or generate) the screen output


In [ ]:
import pathlib, pandas as pd, subprocess, shutil, yaml

ORGANISM = 'yeast'
RUN_SMOKE_SCREEN = False
screen_csv = pathlib.Path(f'results/screening/{ORGANISM}/screening_results.csv')
print('looking for:', screen_csv, '(exists:', screen_csv.exists(), ')')

if not screen_csv.exists() and RUN_SMOKE_SCREEN:
    SMOKE = pathlib.Path('/content/crypticip_smoke')
    PROT  = SMOKE / 'data/proteomes/yeast'
    PROT.mkdir(parents=True, exist_ok=True)
    src = pathlib.Path('tests/fixtures/tiny.pdb')
    for u in ['P00001','P00002','P00003']:
        shutil.copyfile(src, PROT / f'AF-{u}-F1-model_v4.pdb')
    cfg = SMOKE / 'smoke_config.yaml'
    cfg.write_text(yaml.safe_dump({'paths': {
        'data_dir':      str(SMOKE / 'data'),
        'proteomes_dir': str(SMOKE / 'data' / 'proteomes'),
        'cache_dir':     str(SMOKE / 'cache'),
        'results_dir':   'results',
        'reports_dir':   'results/reports',
        'screening_dir': 'results/screening',
        'experimental_dir': 'results/experimental',
    }}))
    subprocess.run(['crypticip','index-proteome','--organism','yeast','--config',str(cfg)], check=False)
    subprocess.run(['crypticip','screen','--organism','yeast','--mode','fast',
                    '--workers','1','--limit','3','--config',str(cfg)], check=False)
elif not screen_csv.exists():
    print('No screen output. Run 02_yeast_screening_colab.ipynb first, mount Drive ',
          'with DRIVE_RESULTS pointing at a previous run, or set RUN_SMOKE_SCREEN=True.')


## Generate experimental plans


In [ ]:
!crypticip experimental-plan --organism {ORGANISM} --top 25


## Inspect mutagenesis plan


In [ ]:
mut_path = pathlib.Path(f'results/experimental/{ORGANISM}/{ORGANISM}_top25_mutagenesis.csv')
if mut_path.exists():
    mut = pd.read_csv(mut_path)
    print('rows:', len(mut))
    mut.head()
else:
    print('No mutagenesis CSV at', mut_path, '— did experimental-plan succeed?')


## Inspect DSF plan


In [ ]:
dsf_path = pathlib.Path(f'results/experimental/{ORGANISM}/{ORGANISM}_top25_dsf.csv')
if dsf_path.exists():
    dsf = pd.read_csv(dsf_path)
    print('rows:', len(dsf))
    dsf.head()
else:
    print('No DSF CSV at', dsf_path)


## Combined experimental ranking


In [ ]:
plan_path = pathlib.Path(f'results/experimental/{ORGANISM}/{ORGANISM}_top25_experimental_plan.csv')
if plan_path.exists():
    plan = pd.read_csv(plan_path)
    plan.head(25)
else:
    print('No combined plan at', plan_path)


## What to download from Colab


In [ ]:
import shutil
src = pathlib.Path(f'results/experimental/{ORGANISM}')
if src.exists():
    out = shutil.make_archive(f'/content/{ORGANISM}_experimental', 'zip', src)
    print('zipped to:', out)
    try:
        from google.colab import files
        files.download(out)
    except Exception as e:
        print('skipping files.download (not on Colab):', e)
else:
    print('No results/experimental/{ORGANISM}/ — run the cells above first.')


## Notes

- The mutagenesis plan suggests target residues for charge-reversal /
  alanine substitution based on the basic residues clustered in each
  predicted pocket.
- The DSF plan suggests fragment / ion conditions per candidate.
- All outputs are CSV under `results/experimental/{ORGANISM}/`. If you
  mounted Drive in the bootstrap, they're already persisted.
